# E0006 — resilient NVFP4 cloud mirror

Purpose: mirror the pinned Hugging Face Nemotron NVFP4 checkpoint into a user-owned Kaggle Model without routing model bytes through the user's PC. The upload child process writes progress to a local log rather than flooding the notebook frontend.

Run in a temporary Kaggle notebook with **Internet ON** and **Accelerator None**. This notebook does not submit to ARC.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

REPO = "nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4"
REVISION = "cc84af2fe71647d87f4486c064f320e1e7535243"
LOCAL = Path("/tmp/nemotron_nvfp4")
HANDLE = "paulomartins87/nemotron-3-5-lightning/pyTorch/30b-a3b-nvfp4"
STATUS = Path("/kaggle/working/e0006_mirror_status.json")
MANIFEST = Path("/kaggle/working/e0006_mirror_manifest.json")
UPLOAD_LOG = Path("/tmp/e0006_kaggle_model_upload.log")
UPLOAD_CHILD = Path("/tmp/e0006_upload_child.py")
MIN_FREE_GIB = 50.0

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

def write_status(stage: str, **extra: object) -> None:
    payload = {
        "experiment": "E0006",
        "purpose": "cloud_mirror_only",
        "stage": stage,
        "source_repo": REPO,
        "source_revision": REVISION,
        "kaggle_handle": HANDLE,
        "unix_time": time.time(),
        **extra,
    }
    STATUS.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(f"[{stage}]", flush=True)

free_gib = shutil.disk_usage("/tmp").free / 1024**3
write_status("PRECHECK", free_tmp_gib=round(free_gib, 3))
if free_gib < MIN_FREE_GIB:
    raise RuntimeError(f"STOP: only {free_gib:.1f} GiB free in /tmp; require {MIN_FREE_GIB:.1f} GiB")

pip_log = Path("/tmp/e0006_pip.log")
with pip_log.open("w", encoding="utf-8") as sink:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub", "huggingface_hub"],
        stdout=sink,
        stderr=subprocess.STDOUT,
        check=True,
    )

from huggingface_hub import snapshot_download

write_status("HF_DOWNLOAD_OR_RESUME", free_tmp_gib=round(shutil.disk_usage("/tmp").free / 1024**3, 3))
model_dir = Path(
    snapshot_download(
        repo_id=REPO,
        revision=REVISION,
        local_dir=str(LOCAL),
        max_workers=4,
    )
)

files = []
total_bytes = 0
for path in sorted(model_dir.rglob("*")):
    if not path.is_file():
        continue
    size = path.stat().st_size
    total_bytes += size
    files.append({"path": str(path.relative_to(model_dir)), "bytes": size})

safetensors = [item for item in files if item["path"].endswith(".safetensors")]
required = {
    "config_json": (model_dir / "config.json").exists(),
    "tokenizer_config": (model_dir / "tokenizer_config.json").exists(),
    "safetensors_present": bool(safetensors),
}
manifest = {
    "source_repo": REPO,
    "source_revision": REVISION,
    "model_dir": str(model_dir),
    "file_count": len(files),
    "safetensor_file_count": len(safetensors),
    "total_bytes": total_bytes,
    "total_gib": round(total_bytes / 1024**3, 3),
    "required_checks": required,
    "files": files,
}
MANIFEST.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
if not all(required.values()):
    write_status("BLOCKED_INCOMPLETE_LOCAL_SNAPSHOT", manifest=str(MANIFEST), required_checks=required)
    raise RuntimeError(f"Incomplete local snapshot: {required}")

write_status("LOCAL_SNAPSHOT_READY", total_gib=manifest["total_gib"], file_count=len(files), manifest=str(MANIFEST))

UPLOAD_CHILD.write_text(
    "import kagglehub\n"
    f"HANDLE = {HANDLE!r}\n"
    f"MODEL_DIR = {str(model_dir)!r}\n"
    f"NOTES = {'Pinned cloud mirror for E0006 ARC Prize 2026 feasibility; source revision ' + REVISION!r}\n"
    "kagglehub.model_upload(HANDLE, MODEL_DIR, version_notes=NOTES)\n"
    "print('UPLOAD_CHILD_RETURNED')\n",
    encoding="utf-8",
)

write_status("KAGGLE_UPLOAD_START", upload_log=str(UPLOAD_LOG))
with UPLOAD_LOG.open("w", encoding="utf-8") as sink:
    process = subprocess.Popen(
        [sys.executable, str(UPLOAD_CHILD)],
        stdout=sink,
        stderr=subprocess.STDOUT,
        text=True,
    )
    started = time.monotonic()
    next_heartbeat = 300.0
    while process.poll() is None:
        time.sleep(10)
        elapsed = time.monotonic() - started
        if elapsed >= next_heartbeat:
            print(f"[KAGGLE_UPLOAD_RUNNING] elapsed={elapsed/60:.1f} min; detailed progress is in {UPLOAD_LOG}", flush=True)
            next_heartbeat += 300.0

returncode = process.returncode
tail = UPLOAD_LOG.read_text(encoding="utf-8", errors="replace")[-12000:] if UPLOAD_LOG.exists() else ""
if returncode != 0:
    write_status("KAGGLE_UPLOAD_FAILED", returncode=returncode, log_tail=tail)
    raise RuntimeError(f"Kaggle upload child failed with return code {returncode}. See {UPLOAD_LOG}")

write_status("KAGGLE_UPLOAD_RETURNED", returncode=returncode, log_tail=tail, total_gib=manifest["total_gib"])
print(f"Mirror upload call returned successfully: {HANDLE}")
print(f"Status: {STATUS}")
print(f"Manifest: {MANIFEST}")
